# V2 Sandwich Overview

Breakdown of all detected sandwiches by structural category:
1. **Standard** — same signer, same owner, single front/back
2. **Multi-split** — multiple front and/or back runs
3. **Diff-signer, same owner** — different signers but same ATA owner
4. **Diff-signer, diff owner** — different signers and different owners

For each category: count, in/cross-block split, profit metrics.

In [ ]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from clickhouse_connect import get_client

# Connect to ClickHouse
load_dotenv(dotenv_path=".env")
if not os.getenv("CLICKHOUSE_HOST"):
    load_dotenv(dotenv_path="../evaluator/.env")

client = get_client(
    host=os.getenv("CLICKHOUSE_HOST", "localhost"),
    port=int(os.getenv("CLICKHOUSE_PORT", 8123)),
    username=os.getenv("CLICKHOUSE_USERNAME", "default"),
    password=os.getenv("CLICKHOUSE_PASSWORD", "sol"),
    database=os.getenv("CLICKHOUSE_DATABASE", "solwich"),
)

print(f"Connected to ClickHouse")

In [ ]:
# Load all sandwiches with classification columns
q = """
SELECT
    sandwichId, slot, crossBlock, consecutive, tokenA, tokenB,
    signerSame, ownerSame,
    multiFrontRun, multiBackRun, frontCount, backCount,
    victimCount, adverseCount,
    profitA, relativeDiffB,
    hasTransfer, hasFrontInlineTransfer, hasDirectTransfer, hasBackInlineTransfer
FROM solwich.sandwiches
LIMIT 1 BY sandwichId
"""
res = client.query(q)
df = pd.DataFrame(res.result_rows, columns=res.column_names)
print(f"Total sandwiches: {len(df):,}")
print(f"Slot range: {df['slot'].min():,} - {df['slot'].max():,}")

In [ ]:
# Classify each sandwich into structural categories
def classify(row):
    if row["signerSame"]:
        if row["multiFrontRun"] or row["multiBackRun"]:
            return "multi_split_same_signer"
        elif row["ownerSame"]:
            return "standard"
        else:
            return "same_signer_diff_owner"
    else:
        if row["ownerSame"]:
            return "diff_signer_same_owner"
        else:
            return "diff_signer_diff_owner"

df["category"] = df.apply(classify, axis=1)
print(df["category"].value_counts())

In [ ]:
# Overall summary by category
def summarize(group, name, total_n=None):
    if total_n is None:
        total_n = len(df)
    n = len(group)
    inblock = (~group["crossBlock"]).sum()
    crossblock = group["crossBlock"].sum()
    consec = group["consecutive"].sum()
    avg_victims = group["victimCount"].mean()
    avg_adverse = group["adverseCount"].mean()

    # Profit metrics: only for tokenA=SOL (comparable units)
    sol_group = group[group["tokenA"] == "SOL"]
    n_sol = len(sol_group)
    profitable = (sol_group["profitA"] > 0).sum() if n_sol > 0 else 0
    total_profit = sol_group["profitA"].sum() if n_sol > 0 else 0
    avg_profit = sol_group["profitA"].mean() if n_sol > 0 else 0
    med_profit = sol_group["profitA"].median() if n_sol > 0 else 0

    return {
        "category": name,
        "count": n,
        "pct": n / total_n * 100,
        "in_block": inblock,
        "in_block_pct": inblock / n * 100 if n > 0 else 0,
        "cross_block": crossblock,
        "cross_block_pct": crossblock / n * 100 if n > 0 else 0,
        "consecutive": consec,
        "consecutive_pct": consec / n * 100 if n > 0 else 0,
        "sol_count": n_sol,
        "sol_profitable": profitable,
        "sol_profitable_pct": profitable / n_sol * 100 if n_sol > 0 else 0,
        "sol_total_profit_SOL": total_profit,
        "sol_avg_profit_SOL": avg_profit,
        "sol_median_profit_SOL": med_profit,
        "avg_victims": avg_victims,
        "avg_adverse": avg_adverse,
    }

categories = [
    "standard",
    "multi_split_same_signer",
    "same_signer_diff_owner",
    "diff_signer_same_owner",
    "diff_signer_diff_owner",
]

rows = []
for cat in categories:
    group = df[df["category"] == cat]
    rows.append(summarize(group, cat))

# Total row
rows.append(summarize(df, "TOTAL"))

summary = pd.DataFrame(rows)
summary

In [ ]:
# Same breakdown but only for tokenA = SOL
df_sol = df[df["tokenA"] == "SOL"].copy()
print(f"SOL-base sandwiches: {len(df_sol):,} ({len(df_sol)/len(df)*100:.1f}% of total)")

rows_sol = []
for cat in categories:
    group = df_sol[df_sol["category"] == cat]
    rows_sol.append(summarize(group, cat, total_n=len(df_sol)))
rows_sol.append(summarize(df_sol, "TOTAL_SOL", total_n=len(df_sol)))

summary_sol = pd.DataFrame(rows_sol)
summary_sol

In [ ]:
# Detailed multi-split analysis
multi = df[df["category"].str.startswith("multi")].copy()
print(f"=== Multi-split sandwiches: {len(multi):,} ===")
print(f"\nFront count distribution:")
print(multi["frontCount"].value_counts().sort_index())
print(f"\nBack count distribution:")
print(multi["backCount"].value_counts().sort_index())
print(f"\nIn-block: {(~multi['crossBlock']).sum():,}, Cross-block: {multi['crossBlock'].sum():,}")
print(f"Profitable: {(multi['profitA'] > 0).sum():,} ({(multi['profitA'] > 0).mean()*100:.1f}%)")

In [ ]:
# Detailed diff-signer analysis
diff_sig = df[df["category"].str.startswith("diff_signer")].copy()
print(f"=== Diff-signer sandwiches: {len(diff_sig):,} ===")
print(f"\nOwner same: {diff_sig['ownerSame'].sum():,}")
print(f"Has transfer: {diff_sig['hasTransfer'].sum():,}")
print(f"Has front inline transfer: {diff_sig['hasFrontInlineTransfer'].sum():,}")
print(f"Has direct transfer: {diff_sig['hasDirectTransfer'].sum():,}")
print(f"Has back inline transfer: {diff_sig['hasBackInlineTransfer'].sum():,}")
print(f"\nIn-block: {(~diff_sig['crossBlock']).sum():,}, Cross-block: {diff_sig['crossBlock'].sum():,}")
print(f"Profitable: {(diff_sig['profitA'] > 0).sum():,} ({(diff_sig['profitA'] > 0).mean()*100:.1f}%)")

In [ ]:
# Profit distribution by category (SOL-base only, for comparable units)
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: count by category
cat_counts = df_sol["category"].value_counts().reindex(categories)
cat_counts.plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("SOL-base Sandwich Count by Category")
axes[0].set_ylabel("Count")
for i, v in enumerate(cat_counts):
    axes[0].text(i, v + 100, f"{v:,}", ha="center", fontsize=9)

# Right: in-block vs cross-block ratio
inblock_pcts = []
for cat in categories:
    g = df_sol[df_sol["category"] == cat]
    inblock_pcts.append((~g["crossBlock"]).mean() * 100 if len(g) > 0 else 0)

bars = axes[1].bar(range(len(categories)), inblock_pcts, color="coral")
axes[1].set_xticks(range(len(categories)))
axes[1].set_xticklabels(categories, rotation=45, ha="right")
axes[1].set_title("In-block Ratio by Category (SOL-base)")
axes[1].set_ylabel("In-block %")
axes[1].set_ylim(0, 100)
for i, v in enumerate(inblock_pcts):
    axes[1].text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Top tokenA distribution (non-SOL)
non_sol = df[df["tokenA"] != "SOL"]
print(f"=== Non-SOL tokenA sandwiches: {len(non_sol):,} ({len(non_sol)/len(df)*100:.1f}%) ===")
print(f"\nTop 15 tokenA:")
print(non_sol["tokenA"].value_counts().head(15).to_string())